# Tests

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
!pip install -e src/

In [3]:
import numpy as np

from src.channels import AWGNChannel, BECChannel
from src.core import PolarCode, PolarEncoder, awgn_frozen_set, bec_frozen_set, sc_decode, sc_decode_llr, bp_decode

In [ ]:
N, K = 16, 8
possible_us: list[list[int]] = [[(i >> shift) & 1 for shift in range(K - 1, -1, -1)] for i in range(2 ** K)]
right_counts: list[int] = []
perfect_count: int = 0 #*number of iterations where the decoder decoded all possible combinations correctly
verbose: int = 0
ITERATIONS: int = 100

BP_ITERATIONS: int = 100 #*bp_decode is iterative and much slower than SC, so it's tested at a smaller scale
bp_possible_us: list[list[int]] = possible_us #*potentially something like possible_us[:32]
bp_right_counts: list[int] = []
bp_perfect_count: int = 0

for i in range(ITERATIONS):
    print("iter", i)
    per_it_right_count: int = 0
    per_it_bp_right_count: int = 0
    for u in possible_us:
        epsilon = 0.05
        frozen_bec = bec_frozen_set(N, K, epsilon)
        codeword_bec = PolarEncoder(PolarCode(N, K, frozen_bec)).encode(u)
        # lrs_bec = BECChannel(epsilon).transmit(codeword_bec, mode='lrs')
        llrs_bec = BECChannel(epsilon).transmit(codeword_bec, mode='llrs') #*Log LRs
        # raw_estimate = sc_decode(frozen_bec, codeword_bec, lrs_bec, BECChannel(epsilon))
        raw_estimate = sc_decode_llr(frozen_bec, llrs_bec)
        clean_estimate = np.array([int(raw_estimate[i]) for i in range(raw_estimate.size) if i not in frozen_bec])
        if np.array_equal(u, clean_estimate):
            if verbose >= 2:
                print(f"Is {u} == {clean_estimate}?", True)
            per_it_right_count += 1
        else:
            if verbose >= 2:
                print(f"Is {u} == {clean_estimate}?", False)

        if i < BP_ITERATIONS and u in bp_possible_us:
            # print(i)
            bp_estimate = bp_decode(frozen_bec, llrs_bec, 10000, True) #*already excludes frozen positions
            if np.array_equal(u, bp_estimate):
                if verbose >= 2:
                    print(f"[BP] Is {u} == {bp_estimate}?", True)
                per_it_bp_right_count += 1
            else:
                if verbose >= 2:
                    print(f"[BP] Is {u} == {bp_estimate}?", False)

    if verbose >= 1:
        print(f"{per_it_right_count} out of {len(possible_us)} were correctly decoded")
    right_counts.append(per_it_right_count)

    if per_it_right_count == len(possible_us):
        perfect_count += 1

    if i < BP_ITERATIONS:
        if verbose >= 1:
            print(f"[BP] {per_it_bp_right_count} out of {len(bp_possible_us)} were correctly decoded")
        bp_right_counts.append(per_it_bp_right_count)

        if per_it_bp_right_count == len(bp_possible_us):
            bp_perfect_count += 1

percentage: int = (perfect_count / ITERATIONS) * 100
print(f"{perfect_count} / {ITERATIONS} SC runs decoded correctly ({percentage:.2f} %)")

bp_percentage: float = (bp_perfect_count / BP_ITERATIONS) * 100
print(f"{bp_perfect_count} / {BP_ITERATIONS} BP runs decoded correctly ({bp_percentage:.2f} %)")


In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

sc_distribution = Counter(right_counts)
bp_distribution = Counter(bp_right_counts)

sc_x = sorted(sc_distribution.keys())
sc_y = [sc_distribution[count] for count in sc_x]

bp_x = sorted(bp_distribution.keys())
bp_y = [bp_distribution[count] for count in bp_x]

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].bar(sc_x, sc_y, width=0.8, color="tab:blue")
axes[0].set_xlabel("Right count")
axes[0].set_ylabel("Number of experiments")
axes[0].set_title(f"SC: Distribution of Right Counts out of {len(possible_us)}")
axes[0].set_xticks(sc_x)
axes[0].grid(axis="y", alpha=0.3)

axes[1].bar(bp_x, bp_y, width=0.8, color="tab:orange")
axes[1].set_xlabel("Right count")
axes[1].set_ylabel("Number of experiments")
axes[1].set_title(f"BP: Distribution of Right Counts out of {len(bp_possible_us)}")
axes[1].set_xticks(bp_x)
axes[1].grid(axis="y", alpha=0.3)

#*Denominators differ (len(possible_us) vs len(bp_possible_us)), so compare on a % scale
sc_pct = [count / len(possible_us) * 100 for count in right_counts]
bp_pct = [count / len(bp_possible_us) * 100 for count in bp_right_counts]

#*Fixed range keeps every bin inside [0, 100] so bars are never cropped by the axis
axes[2].hist(sc_pct, bins=20, range=(0, 100), color="tab:blue", alpha=0.6, label="SC")
axes[2].hist(bp_pct, bins=20, range=(0, 100), color="tab:orange", alpha=0.6, label="BP")
axes[2].set_xlabel("Correct decodes (%)")
axes[2].set_ylabel("Number of experiments")
axes[2].set_title("SC vs BP: Distribution of Correct-Decode Percentage")
axes[2].set_xlim(0, 100)
axes[2].grid(axis="y", alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.show()


In [ ]:
np.float16(1.0 / 0.0)

In [ ]:
mat = np.array([[1, 2], [3, 4]])
mat = np.zeros([4, 5])
mat
arr1 = np.array([0, 1, 0])
arr2 = np.array([0, 1, 1])
for u_i, i in enumerate(arr2):
    print(u_i)
(arr1 == arr2).all()

In [18]:
graph = [[["t", i], ["b", 8 - i]] for i in range(8)]
graph[0]

[['t', 0], ['b', 8]]

In [50]:
u = [1, 0, 0, 0, 1, 1, 0, 1]
frozen_bec = bec_frozen_set(16, 8, 0.5)
codeword_bec = PolarEncoder(PolarCode(16, 8, frozen_bec)).encode(u)
llrs_bec = BECChannel(0.5).transmit(codeword_bec, mode='llrs') #*Log LRs
bp_estimate = bp_decode(frozen_bec, llrs_bec, 10000, True) #*already excludes frozen positions
